# Abdomen 3D CT Viewer 
CT hacimlerini (NIfTI), 3D bounding box projeksiyonlarını ve nnDetection instance maskelerini interaktif görselleştirir.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Proje yolları ──────────────────────────────────────────────
DATA_ROOT  = Path(os.environ.get("TR_ABDOMEN_BASE",  "."))
CODE       = Path(os.environ.get("TR_ABDOMEN_CODE",  "."))
sys.path.insert(0, str(CODE))

from src.config  import OUT_DIR, SPLIT_DIR, SUPER_CLASSES
from src.splits  import load_fold

NND_ROOT   = OUT_DIR / "nndet"
NIFTI_DIR  = NND_ROOT / "nifti"
LABELS_DIR = NND_ROOT / "raw" / "Task100_Abdomen" / "raw_splitted" / "labelsTr"
MANIFEST   = pd.read_csv(SPLIT_DIR / "manifest.csv")

# ── Sınıf renkleri (6 sınıf) ──────────────────────────────────
CLASS_COLORS = [
    "#e74c3c",  # 0 acute_cholecystitis
    "#3498db",  # 1 kidney_ureter_stone
    "#2ecc71",  # 2 acute_pancreatitis
    "#f39c12",  # 3 aortic_aneurysm_dissection
    "#9b59b6",  # 4 acute_appendicitis
    "#1abc9c",  # 5 acute_diverticulitis
]

def hex_to_rgba(h, alpha=0.35):
    h = h.lstrip("#")
    r, g, b = [int(h[i:i+2], 16)/255 for i in (0, 2, 4)]
    return (r, g, b, alpha)

print("Yollar OK:", NIFTI_DIR.exists(), LABELS_DIR.exists())
print("Sınıflar:", SUPER_CLASSES)

In [ ]:
# ── Yardımcı fonksiyonlar ──────────────────────────────────────
import pydicom

def _2d_iou(a, b):
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    iw = max(0.0, min(ax2,bx2)-max(ax1,bx1))
    ih = max(0.0, min(ay2,by2)-max(ay1,by1))
    inter = iw*ih
    return inter / max((ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter, 1e-6)

def load_ct_volume(case, nifti_dir, wc=50, ww=400):
    p = nifti_dir / f"case_{case:05d}_0000.nii.gz"
    if not p.exists(): return None
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(p))).astype(np.float32)
    lo, hi = wc - ww/2, wc + ww/2
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def load_instance_mask(case, labels_dir):
    """NIfTI segmentasyon maskesini yükler."""
    nii_p  = labels_dir / f"case_{case:05d}.nii.gz"
    json_p = labels_dir / f"case_{case:05d}.json"
    if not nii_p.exists(): return None, {}
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_p))).astype(np.uint8)
    inst2cls = {}
    if json_p.exists():
        info = json.loads(json_p.read_text())
        for k, v in info.get("instances", {}).items():
            # v formatı: {"class": 1}  veya doğrudan int (eski format)
            cls = v["class"] if isinstance(v, dict) else int(v)
            inst2cls[int(k)] = int(cls)
    return mask, inst2cls

def any_masks_exist(labels_dir):
    return any(labels_dir.glob("*.nii.gz"))

def case_metadata(case, manifest, super_classes):
    sub = manifest[manifest["case"] == case]
    cls_set = set()
    for _, row in sub.iterrows():
        bb = str(row.get("bboxes", ""))
        if bb.strip() in ("", "nan"): continue
        for seg in bb.split("|"):
            p = seg.split(",")
            if p:
                try: cls_set.add(int(p[0]))
                except: pass
    names = [super_classes[c] for c in sorted(cls_set) if c < len(super_classes)]
    return f"Vaka {case} | {len(sub)} kesit | {', '.join(names) or 'annotasyon yok'}"

In [ ]:
from tqdm import tqdm

def build_bbox_masks(nifti_dir, labels_dir, overwrite=False):
    """
    labelsTr'deki her JSON için instance segmentasyon maskesi (.nii.gz) üretir.
    NIfTI header'ından boyut/metadata okunur — piksel verisi yüklenmez (hızlı).
    """
    json_files = sorted(labels_dir.glob("*.json"))
    ok = skip = err = 0

    for jf in tqdm(json_files, desc="Bbox→Maske"):
        case_num = int(jf.stem.split("_")[1])
        out_nii  = labels_dir / f"case_{case_num:05d}.nii.gz"

        if out_nii.exists() and not overwrite:
            skip += 1
            continue

        nii_src = nifti_dir / f"case_{case_num:05d}_0000.nii.gz"
        if not nii_src.exists():
            err += 1
            continue

        try:
            data     = json.loads(jf.read_text())
            boxes    = data.get("boxes", [])
            inst_ids = [int(k) for k in data.get("instances", {}).keys()]

            # Sadece header oku — piksel verisi yüklenmez
            reader = sitk.ImageFileReader()
            reader.SetFileName(str(nii_src))
            reader.ReadImageInformation()
            W, H, D = reader.GetSize()   # SimpleITK: (W, H, D)

            mask_arr = np.zeros((D, H, W), dtype=np.uint8)

            for inst_id, box in zip(inst_ids, boxes):
                x1, y1, z1, x2, y2, z2 = [int(v) for v in box]
                mask_arr[max(0,z1):min(D,z2+1),
                         max(0,y1):min(H,y2+1),
                         max(0,x1):min(W,x2+1)] = inst_id

            mask_img = sitk.GetImageFromArray(mask_arr)
            mask_img.SetOrigin(reader.GetOrigin())
            mask_img.SetSpacing(reader.GetSpacing())
            mask_img.SetDirection(reader.GetDirection())
            sitk.WriteImage(mask_img, str(out_nii))
            ok += 1

        except Exception as e:
            print(f"  [hata] case {case_num}: {e}")
            err += 1

    print(f"\n✓ Oluşturuldu: {ok},  ⏭ Atlandı: {skip},  ✗ Hata: {err}")
    print(f"Toplam maske: {sum(1 for _ in labels_dir.glob('*.nii.gz'))}")


build_bbox_masks(NIFTI_DIR, LABELS_DIR, overwrite=False)
MASKS_AVAILABLE = any_masks_exist(LABELS_DIR)
print(f"Maske durumu: {'✓ hazır' if MASKS_AVAILABLE else '✗ hâlâ yok'}")

In [ ]:

MASKS_AVAILABLE = any_masks_exist(LABELS_DIR)
print("Helper fonksiyonlar hazır.")
print(f"Segmentasyon maskesi: {'✓ mevcut' if MASKS_AVAILABLE else '✗ yok — nnDetection inference bekleniyor'}")

In [ ]:
!pip install ipywidgets

In [ ]:
from ipywidgets import IntSlider, Dropdown, HBox, VBox, Output, ToggleButton, HTML
from IPython.display import display
import warnings; warnings.filterwarnings("ignore")

EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))

_fold = 0
_train_cases = set(load_fold(_fold, "train"))
_val_cases   = set(load_fold(_fold, "val"))
_all_cases   = sorted(_train_cases | _val_cases)

_case_classes = {}
for _c in _all_cases:
    _sub = MANIFEST[MANIFEST["case"] == _c]
    _cls = set()
    for _, _row in _sub.iterrows():
        _bb = str(_row.get("bboxes", ""))
        if _bb.strip() in ("", "nan"): continue
        for _seg in _bb.split("|"):
            _p = _seg.split(",")
            if _p:
                try: _cls.add(int(_p[0]))
                except: pass
    _case_classes[_c] = _cls


def case_image_id_to_z_fast(case, egitim_dir):
    cd = egitim_dir / str(case)
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(cd))
    if not series_ids:
        return {}
    names = reader.GetGDCMSeriesFileNames(str(cd), series_ids[0])
    result = {}
    for z, name in enumerate(names):
        stem = Path(name).stem
        if stem.startswith("."):
            continue
        try:
            result[int(stem)] = z
        except ValueError:
            pass
    return result


def abdomen_viewer(fold=0, wc=50, ww=400):
    train_cases = set(load_fold(fold, "train"))
    val_cases   = set(load_fold(fold, "val"))
    all_cases   = sorted(train_cases | val_cases)

    split_dd = Dropdown(description="Split:", value="Tümü",
                        options=["Tümü","Train","Val"], layout={"width":"160px"})
    class_dd = Dropdown(description="Sınıf:", value="Tümü",
                        options=["Tümü"]+list(SUPER_CLASSES), layout={"width":"280px"})
    case_dd  = Dropdown(description="Vaka:", layout={"width":"200px"})
    slider   = IntSlider(description="Kesit:", continuous_update=False,
                         layout={"width":"500px"})
    mask_btn = ToggleButton(value=False, description="Maske Göster",
                            button_style="info" if MASKS_AVAILABLE else "",
                            disabled=not MASKS_AVAILABLE,
                            layout={"width":"160px"})
    box_btn  = ToggleButton(value=True, description="Kutu Göster",
                            button_style="warning", layout={"width":"160px"})
    info_out = Output()
    img_out  = Output()

    mask_note = HTML(
        value="<span style='color:#e67e22;font-size:11px'>⚠ Maske yok — nnDetection inference bekleniyor</span>"
        if not MASKS_AVAILABLE else ""
    )

    def filtered_cases():
        sp = split_dd.value; cn = class_dd.value
        cases = all_cases
        if sp == "Train": cases = [c for c in cases if c in train_cases]
        elif sp == "Val": cases = [c for c in cases if c in val_cases]
        if cn != "Tümü":
            ci = SUPER_CLASSES.index(cn)
            cases = [c for c in cases if ci in _case_classes.get(c, set())]
        return cases

    def update_cases(change):
        cases = filtered_cases()
        case_dd.options = cases
        if cases: case_dd.value = cases[0]

    _cache = {}   # case → (vol, msk, inst2cls, boxes, z_to_imgid)

    def get_data(case):
        if case not in _cache:
            vol      = load_ct_volume(case, NIFTI_DIR, wc, ww)
            msk, i2c = load_instance_mask(case, LABELS_DIR)
            boxes      = []
            z_to_imgid = {}
            try:
                z_map      = case_image_id_to_z_fast(case, EGITIM_DIR)
                z_to_imgid = {z: img_id for img_id, z in z_map.items()}
                sub        = MANIFEST[(MANIFEST["case"] == case) &
                                       (MANIFEST["bboxes"].fillna("").astype(str) != "")]
                items = []
                for _, row in sub.iterrows():
                    z = z_map.get(int(row["image_id"]))
                    if z is None: continue
                    for bb_str in str(row["bboxes"]).split("|"):
                        parts = bb_str.split(",")
                        if len(parts) < 5: continue
                        sid = int(parts[0]); x1,y1,x2,y2 = map(int, parts[1:5])
                        items.append((sid, x1, y1, x2, y2, z))
                for sid in set(it[0] for it in items):
                    cls_items = sorted([it for it in items if it[0]==sid], key=lambda x: x[5])
                    used = set()
                    for i, it in enumerate(cls_items):
                        if i in used: continue
                        grp = [it]; used.add(i)
                        for j in range(i+1, len(cls_items)):
                            if j in used: continue
                            last = grp[-1]
                            if cls_items[j][5]-last[5] <= 2 and _2d_iou(last[1:5], cls_items[j][1:5]) >= 0.3:
                                grp.append(cls_items[j]); used.add(j)
                            elif cls_items[j][5]-last[5] > 2:
                                break
                        boxes.append({"class": sid,
                                      "x1": min(g[1] for g in grp), "y1": min(g[2] for g in grp),
                                      "x2": max(g[3] for g in grp), "y2": max(g[4] for g in grp),
                                      "z1": min(g[5] for g in grp), "z2": max(g[5] for g in grp)})
            except Exception as e:
                print(f"Kutu yüklenemedi: {e}")
            _cache[case] = (vol, msk, i2c, boxes, z_to_imgid)
        return _cache[case]

    def draw(z, case, show_mask, show_boxes):
        vol, msk, i2c, boxes, z_to_imgid = get_data(case)
        if vol is None:
            with img_out:
                img_out.clear_output(wait=True)
                print(f"⚠️  Vaka {case} için NIfTI bulunamadı.")
            return

        img_id   = z_to_imgid.get(z, "—")
        has_mask = show_mask and msk is not None
        n_cols   = 1 + int(show_boxes) + int(has_mask)
        fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5), facecolor="#111")
        if n_cols == 1: axes = [axes]
        col = 0

        axes[col].imshow(vol[z], cmap="gray", vmin=0, vmax=1)
        axes[col].set_title(f"CT  z={z+1}/{vol.shape[0]}  img_id={img_id}",
                            color="white", fontsize=9)
        axes[col].axis("off"); col += 1

        if show_boxes:
            axes[col].imshow(vol[z], cmap="gray", vmin=0, vmax=1)
            sl_boxes = [b for b in boxes if b["z1"] <= z <= b["z2"]]
            for b in sl_boxes:
                c_idx = b["class"]
                color = CLASS_COLORS[c_idx % len(CLASS_COLORS)]
                rect  = mpatches.Rectangle(
                    (b["x1"], b["y1"]), b["x2"]-b["x1"], b["y2"]-b["y1"],
                    linewidth=2, edgecolor=color, facecolor="none")
                axes[col].add_patch(rect)
                label = SUPER_CLASSES[c_idx] if c_idx < len(SUPER_CLASSES) else str(c_idx)
                axes[col].text(b["x1"], max(b["y1"]-3, 0), label,
                               color=color, fontsize=6, weight="bold",
                               bbox=dict(facecolor="black", alpha=0.4, pad=1))
            axes[col].set_title(f"3D Kutular  ({len(sl_boxes)} kutu)  img_id={img_id}",
                                color="white", fontsize=9)
            axes[col].axis("off"); col += 1

        if has_mask:
            axes[col].imshow(vol[z], cmap="gray", vmin=0, vmax=1)
            msk_z = msk[z]
            for inst_id, cls in i2c.items():
                region = (msk_z == inst_id)
                if not region.any(): continue
                color = CLASS_COLORS[cls % len(CLASS_COLORS)]
                r, g, b_ = [int(color.lstrip("#")[i:i+2], 16)/255 for i in (0,2,4)]
                rgba = np.zeros((*region.shape, 4), dtype=np.float32)
                rgba[region] = [r, g, b_, 0.4]
                axes[col].imshow(rgba)
                axes[col].contour(region.astype(float), levels=[0.5],
                                  colors=[color], linewidths=1.5)
            axes[col].set_title(f"Maske  img_id={img_id}", color="white", fontsize=9)
            axes[col].axis("off")

        patches = [mpatches.Patch(color=CLASS_COLORS[i], label=SUPER_CLASSES[i])
                   for i in range(len(SUPER_CLASSES))]
        fig.legend(handles=patches, loc="lower center", ncol=3,
                   fontsize=7, facecolor="#222", labelcolor="white", framealpha=0.8)

        tag = "Train" if case in train_cases else "Val"
        fig.suptitle(case_metadata(case, MANIFEST, SUPER_CLASSES) + f"  [{tag}]",
                     color="white", fontsize=10, fontweight="bold")
        plt.tight_layout(rect=[0, 0.1, 1, 1])
        with img_out:
            img_out.clear_output(wait=True); plt.show()

    def on_change(change):
        if case_dd.value is None: return
        draw(slider.value, case_dd.value, mask_btn.value, box_btn.value)

    def on_case_change(change):
        case = case_dd.value
        if case is None: return
        with info_out:
            info_out.clear_output()
            print(f"Vaka {case} yükleniyor...")
        vol, *_ = get_data(case)
        n_slices = vol.shape[0] if vol is not None else "?"
        if vol is not None:
            slider.max   = vol.shape[0] - 1
            slider.value = vol.shape[0] // 2
        with info_out:
            info_out.clear_output()
            sub        = MANIFEST[MANIFEST["case"] == case]
            ann_slices = (sub["bboxes"].fillna("").astype(str) != "").sum()
            cls_names  = case_metadata(case, MANIFEST, SUPER_CLASSES).split("|")[-1].strip()
            print(f"Vaka {case}  |  Toplam: {n_slices} kesit  |  Annotasyonlu: {ann_slices} kesit")
            print(f"  Sınıflar : {cls_names}")
            print(f"  Split    : {'Train' if case in train_cases else 'Val'}")
            print(f"  NIfTI    : {'✓' if (NIFTI_DIR / f'case_{case:05d}_0000.nii.gz').exists() else '✗'}")
            print(f"  Maske    : {'✓' if (LABELS_DIR / f'case_{case:05d}.nii.gz').exists() else '✗'}")
        draw(slider.value, case, mask_btn.value, box_btn.value)

    for w in [split_dd, class_dd]: w.observe(update_cases, names="value")
    case_dd.observe(on_case_change, names="value")
    slider.observe(on_change, names="value")
    mask_btn.observe(on_change, names="value")
    box_btn.observe(on_change, names="value")

    display(VBox([
        HBox([split_dd, class_dd]),
        HBox([case_dd, mask_btn, box_btn, mask_note]),
        info_out, slider, img_out,
    ]))
    update_cases(None)

In [ ]:
abdomen_viewer(fold=0, wc=50, ww=400)